# 군 장비 정비 보고서 고장 유형 분류 (Problem#3)
메인 v3: 방해 문장 제거 → 핵심 증상 문장 추출 → 글자 n-gram TF-IDF + 계통 키워드 사전 피처 → LogisticRegression.

검증: 같은 핵심 증상(문구)이 train에 여러 번 반복되고 test는 모두 처음 보는 표현이므로, **핵심 증상 단위 StratifiedGroupKFold**로 평가한다(무작위 KFold는 0.99 이상으로 과대평가됨).

검증 환경: Python 3.9 / numpy 1.23.5 / pandas 1.5.2 / scikit-learn 1.1.3 (대회 서버 버전), CPU 2코어. 추가 패키지 없음.

In [1]:
# 1. 환경 및 설정
import os, re, time, platform, warnings
import numpy as np, pandas as pd, sklearn
from scipy import sparse
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.metrics import f1_score, confusion_matrix, classification_report
warnings.filterwarnings('ignore')
print('python', platform.python_version(), '| numpy', np.__version__, '| pandas', pd.__version__, '| sklearn', sklearn.__version__)

SEED = 42
tcols, target, idcol = ['report'], 'fault_type', 'id'
CLASSES = ['COMMUNICATION', 'COOLING', 'ELECTRICAL', 'ENGINE', 'HYDRAULIC', 'MOBILITY']
CANDIDATES = ['data', '.', '/content/drive/MyDrive/MAICON준비/2026/prob_03/data', '/content/data']
DATA_DIR = next(d for d in CANDIDATES if os.path.exists(os.path.join(d, 'train.csv')))
print('DATA_DIR =', DATA_DIR)

python 3.9.25 | numpy 1.23.5 | pandas 1.5.2 | sklearn 1.1.3
DATA_DIR = data


In [2]:
# 2. 데이터 로드
train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
print(train.shape, test.shape)
print(train[target].value_counts().to_dict())
assert train[idcol].is_unique and test[idcol].is_unique
def get_text(d):
    s = d[tcols[0]].fillna('').astype(str)
    for c in tcols[1:]:
        s = s + ' ' + d[c].fillna('').astype(str)
    return s

(540, 3) (180, 2)
{'HYDRAULIC': 90, 'COMMUNICATION': 90, 'MOBILITY': 90, 'ENGINE': 90, 'COOLING': 90, 'ELECTRICAL': 90}


In [3]:
# 3. 정제: 다른 계통의 '의심했으나 정상' 같은 방해 문장, 부가 점검 문장, 장비명·서식 머리말 제거 -> 핵심 증상 문장
DISTRACT = [r'의심했으나', r'보고했으나', r'반복\s*점검했지만', r'함께\s*확인했으나', r'우선\s*점검한\s*결과',
            r'가능성은\s*낮으며', r'시험을\s*반복했지만', r'뒷받침하는\s*징후는\s*발견되지\s*않', r'^초기\s*증상은']
INTRO = r'^(참고로|추가\s*확인\s*결과|별도\s*점검에서|다른\s*점검\s*항목\s*중|동시에\s*(점검|확인)한\s*항목(에서는|\s*중)|별도\s*확인\s*항목은|\[기타\]|한편)'
TAILS = [r'외기\s*온도는\s*정상\s*범위였음', r'주변\s*환경\s*조건은\s*정상\s*범위였다', r'증상은\s*2회\s*연속\s*재현됨',
         r'동일\s*현상이\s*재시험에서도\s*반복되었다', r'짧은\s*확인\s*운전에서도\s*같은\s*현상이\s*나타났다',
         r'정비\s*후\s*첫\s*운용에서\s*확인됨', r'\d+분\s*점검\s*중\s*동일\s*현상이\s*반복됨',
         r'정지\s*후\s*다시\s*시작하면\s*일시적으로\s*정상처럼\s*보인다', r'동일\s*조건에서\s*두\s*차례\s*반복해도\s*같은\s*현상이\s*나타났다',
         r'외부\s*충격\s*흔적은\s*발견되지\s*않음', r'이라고\s*보고함']
NORMAL = r'(정상|양호|통과|없음|없었다|없다|만족|재현되지\s*않았다|확인되지\s*않았다|기준\s*범위|허용\s*범위|점등)\s*(임|이다|였다|이었다|다|함|됨|했다|했음)?$'
CONTRAST = r'(으나|지만|외에|있고|이후|났고|[가-힣]나\s)'
WRAP = [r'^\[현상\]\s*', r'^\[점검\s*메모\]\s*\S.*?에서\s+', r'^정비\s*메모\s*:\s*[^:]*:\s*', r'^현장\s*정비\s*기록\s*:\s*\S.*?에서\s+',
        r'^운용자\s*인계\s*(사항|기록)\([^)]*\)\s*:\s*', r'^.*?운용\s*후\s*(정비반\s*)?보고\s*:\s*', r'^.*?(재현\s*)?시험\s*결과,?\s*',
        r'^.*?재현\s*시험에서\s*', r'^.*?점검\s*결과,?\s*', r'^.*?다음\s*사항을\s*확인함\s*', r'^점검\s*대상은\s*.*?운용자가\s*']

def split_sents(s):
    s = s.replace('—', ':')
    parts = re.split(r'(?<=[.])\s+|\.$|\s(?=\[(?:현상|대상|기타)\])', s)
    return [p.strip(' .') for p in parts if p and p.strip(' .')]

def clean(s):
    keep = []
    for p in split_sents(s):
        q = p
        for t in TAILS: q = re.sub(t, ' ', q)
        q = re.sub(r'\s+\S+(과|와)\s+\S+\s+시험을\s+반복했지만.*$', '', q)
        q = re.sub(r'\s+', ' ', q).strip(' .')
        if not q or re.search(r'다음\s*사항을\s*확인함$', q): continue
        if any(re.search(d, q) for d in DISTRACT) or re.search(INTRO, q) or q.startswith('[대상]'): continue
        if re.search(NORMAL, q) and not re.search(CONTRAST, q) and len(q) < 40: continue
        keep.append(q)
    return ' '.join(keep) if keep else s           # 다 지워지면 원문 유지

def core(s):
    t = clean(s)
    for w in WRAP:
        t2 = re.sub(w, '', t, count=1)
        if t2 != t and len(t2) >= 15:
            t = t2; break
    return re.sub(r'\s+', ' ', t).strip(' .')

X_raw, y, T_raw = get_text(train), train[target].values, get_text(test)
X, X_test = X_raw.map(core), T_raw.map(core)
groups = X.values                                   # 같은 핵심 증상 = 같은 그룹
print('train 핵심 증상 종류', X.nunique(), '| test', X_test.nunique(), '| test 중 train과 같은 문장', int(X_test.isin(X).sum()))
print('핵심 증상 -> 라벨 일관성', (pd.Series(y).groupby(X.values).nunique() == 1).mean())
for a, b in zip(T_raw.head(3), X_test.head(3)): print(a, '\n  ->', b)

train 핵심 증상 종류 90 | test 95 | test 중 train과 같은 문장 0
핵심 증상 -> 라벨 일관성 1.0
현장 지원 장비에 대한 재현 시험에서 핸들을 움직여도 진행 방향이 바뀌지 않아 조향 제어를 상실했다. 동일 현상이 재시험에서도 반복되었다. 한편 별도 실시한 전원 점검은 정상 범위였다 
  -> 핸들을 움직여도 진행 방향이 바뀌지 않아 조향 제어를 상실했다
정비 메모 — 다목적 지원 차량: 첫 기동에 여러 번의 시도가 필요하지만 일단 가동되면 저부하 운전은 가능하다. 동시에 점검한 항목에서는 외관상 큰 충격 흔적은 없다 
  -> 첫 기동에 여러 번의 시도가 필요하지만 일단 가동되면 저부하 운전은 가능하다
정비 메모 — 정비 지원 장비: 가벼운 부하는 처리되지만 큰 부하에서는 충분한 압력이 유지되지 않는다. 동시에 점검한 항목에서는 운용 전 점검표의 나머지 항목은 통과했다 
  -> 가벼운 부하는 처리되지만 큰 부하에서는 충분한 압력이 유지되지 않는다


In [4]:
# 4. 계통 키워드 사전 피처 (일반 도메인 용어, 계통별 등장 횟수)
LEX = {
 'ENGINE': ['엔진', '시동', '기동', '회전', '출력', '연료', '연소', '크랭킹', '배기', '공회전', '윤활', '가속', '힘이', '동력', '마운트', '기계음', '거친 소리', '거친 소음', '회전수'],
 'ELECTRICAL': ['전원', '전압', '배터리', '축전', '회로', '퓨즈', '차단', '배선', '단자', '표시', '조명', '램프', '계기', '충전', '전류', '커넥터', '전기', '전력'],
 'HYDRAULIC': ['유압', '압력', '작동유', '펌프', '실린더', '호스', '배관', '피팅', '누유', '액추에이터', '유체', '구동부', '유분', '누설', '젖어', '습윤', '액면', '저압'],
 'COMMUNICATION': ['통신', '송신', '수신', '무선', '안테나', '채널', '신호', '음성', '호출', '링크', '데이터', '전송', '송수신', '접속', '감도', '자료 교환', '시간 초과', '세션', '단말', '상호 연결', '발신', '연결이'],
 'MOBILITY': ['조향', '핸들', '제동', '브레이크', '타이어', '바퀴', '차륜', '휠', '주행', '감속', '트레드', '쏠림', '공기압', '자체 이동', '이동이', '운행', '차체', '직진', '한쪽으로'],
 'COOLING': ['냉각', '냉각수', '온도', '과열', '팬', '라디에이터', '열', '흡입', '배출구', '풍량', '증기', '뜨거', '식힌', '저장조', '보조 탱크', '보호 경보', '보호 경고', '보호 기능', '보호 모드', '자동 정지', '측정값이 허용 상한', '측정값이 계속 상승'],
}
class Lex(BaseEstimator, TransformerMixin):
    def __init__(self, w=1.0): self.w = w
    def fit(self, X, y=None): return self
    def transform(self, X):
        M = np.array([[sum(s.count(k) for k in ks) for ks in LEX.values()] for s in X], dtype=float)
        return sparse.csr_matrix(np.log1p(M) * self.w)

def make_model(use_lex=True):
    feats = [('tfidf', TfidfVectorizer(analyzer='char_wb', ngram_range=(1, 3), sublinear_tf=True))]
    if use_lex: feats.append(('lex', Lex(1.0)))
    return Pipeline([('feat', FeatureUnion(feats)),
                     ('clf', LogisticRegression(C=10, max_iter=2000, class_weight='balanced'))])

def make_baseline():   # 초안: 원문 글자(2,4) TF-IDF + LR
    return Pipeline([('vec', TfidfVectorizer(analyzer='char', ngram_range=(2, 4))),
                     ('clf', LogisticRegression(max_iter=1000, class_weight='balanced'))])

In [5]:
# 5. 교차검증: 핵심 증상 단위 그룹 5-fold x 3 seed (Macro F1). 무작위 5-fold는 참고용
def cv_f1(make, Xs, kind='group', seeds=(0, 1, 2)):
    scores, last = [], None
    for sd in seeds:
        if kind == 'group':
            sp = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=sd).split(Xs, y, groups)
        else:
            sp = StratifiedKFold(n_splits=5, shuffle=True, random_state=sd).split(Xs, y)
        pr = np.empty(len(y), dtype=object)
        for a, b in sp:
            m = make(); m.fit(Xs.iloc[a], y[a]); pr[b] = m.predict(Xs.iloc[b])
        scores.append(f1_score(y, pr, average='macro')); last = pr
    return np.mean(scores), np.std(scores), last

t0 = time.time()
res = {}
res['v1 초안 (원문 char(2,4) LR), 무작위 CV'] = cv_f1(make_baseline, X_raw, 'random')
res['v1 초안 (원문 char(2,4) LR)']           = cv_f1(make_baseline, X_raw)
res['v2 핵심 문장 + char_wb(1,3) LR']        = cv_f1(lambda: make_model(False), X)
res['v3 핵심 문장 + char_wb(1,3) + 사전 LR']  = cv_f1(lambda: make_model(True), X)
for k, (mu, sd, _) in res.items(): print('%-38s Macro F1 %.4f ± %.4f' % (k, mu, sd))
print('CV time %.0fs' % (time.time() - t0))
oof = res['v3 핵심 문장 + char_wb(1,3) + 사전 LR'][2]
print(pd.DataFrame(confusion_matrix(y, oof, labels=CLASSES), index=CLASSES, columns=CLASSES))

v1 초안 (원문 char(2,4) LR), 무작위 CV        Macro F1 0.9994 ± 0.0009
v1 초안 (원문 char(2,4) LR)                Macro F1 0.4315 ± 0.0081
v2 핵심 문장 + char_wb(1,3) LR             Macro F1 0.7894 ± 0.0153
v3 핵심 문장 + char_wb(1,3) + 사전 LR        Macro F1 0.9889 ± 0.0000
CV time 14s
               COMMUNICATION  COOLING  ELECTRICAL  ENGINE  HYDRAULIC  MOBILITY
COMMUNICATION             90        0           0       0          0         0
COOLING                    0       90           0       0          0         0
ELECTRICAL                 0        0          90       0          0         0
ENGINE                     0        0           0      84          0         6
HYDRAULIC                  0        0           0       0         90         0
MOBILITY                   0        0           0       0          0        90


In [6]:
# 6. 전체 학습 -> test 예측 -> submission.csv (test.csv의 id 순서 그대로)
model = make_model(True).fit(X, y)
assert list(model.classes_) == CLASSES
pred = model.predict(X_test)
sub = pd.DataFrame({idcol: test[idcol].values, target: pred})
sub.to_csv('submission.csv', index=False)

# 7. 제출 형식 검증 (저장 후 다시 읽기)
chk = pd.read_csv('submission.csv')
assert list(chk.columns) == [idcol, target]
assert len(chk) == len(test) and chk[idcol].is_unique
assert chk[idcol].tolist() == test[idcol].tolist()
assert chk[target].isin(CLASSES).all() and chk[target].notnull().all()
print(chk.shape); print(chk.head())
print('train 비율', pd.Series(y).value_counts(normalize=True).sort_index().round(3).to_dict())
print('test 예측 비율', chk[target].value_counts(normalize=True).sort_index().round(3).to_dict())

(180, 2)
        id     fault_type
0  TST0001       MOBILITY
1  TST0002         ENGINE
2  TST0003      HYDRAULIC
3  TST0004     ELECTRICAL
4  TST0005  COMMUNICATION
train 비율 {'COMMUNICATION': 0.167, 'COOLING': 0.167, 'ELECTRICAL': 0.167, 'ENGINE': 0.167, 'HYDRAULIC': 0.167, 'MOBILITY': 0.167}
test 예측 비율 {'COMMUNICATION': 0.161, 'COOLING': 0.15, 'ELECTRICAL': 0.172, 'ENGINE': 0.172, 'HYDRAULIC': 0.178, 'MOBILITY': 0.167}
